In [1]:
import os
import astropy.units as u
from astropy.coordinates import FK5, SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astroquery.astrometry_net import AstrometryNet

import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import simple_norm
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

import io
import requests
import astropy.units as u
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
from joblib import Parallel, delayed

from astroquery.sdss import SDSS
from astropy import coordinates as coords
import pandas as pd
from astropy.coordinates import SkyCoord, FK5
import astropy.units as u
import matplotlib
matplotlib.use('Agg')


import io
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import wcsaxes
import requests

In [7]:
GEHR_list = pd.read_csv('NewDavidGEHRList.csv',index_col=False)
aux_host = pd.read_csv('aux_host.csv',index_col=False)
host_galaxies = np.unique(GEHR_list['GALAXY'])
url = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits"

In [9]:
search_status = 'galaxy_match_status.txt'
if not os.path.exists(search_status):
    print(f"Archivo no encontrado. Creando {search_status} para guardar valores... \n")
    with open(search_status, 'w') as f:
        pass




for host in host_galaxies:

    status_list = []
    with open(search_status, 'r') as f:
        for linea in f:
            status_list.append(linea.strip())

    if f"{host}" not in status_list:

        print(f"{host}: \n" )
        selec = GEHR_list[GEHR_list['GALAXY'] == host]
        host_info_aux = aux_host[aux_host['GALAXY'] == host]
        print(f"{host_info_aux.iloc[0]['SURVEY']} , {host_info_aux.iloc[0]['RA']} , {host_info_aux.iloc[0]['DEC']} \n" )
        for i in range(len(selec)):
            print(f"{selec.iloc[i]['CLASS']}	{selec.iloc[i]['RA']}	{selec.iloc[i]['DEC']} \n")
        #host_coordinates = SkyCoord(
        #    ra=host_info_aux.iloc[0]['RA'], 
        #    dec=host_info_aux.iloc[0]['DEC'], 
        #    unit=(u.hourangle, u.deg), # RA en dec, DEC en dec
        #    frame='icrs', 
        #    equinox='J2000'
        #)
        ra_real = host_info_aux.iloc[0]['RA'] #host_coordinates.ra.deg          
        dec_real = host_info_aux.iloc[0]['DEC'] #host_coordinates.dec.deg                
        fov_en_grados = (selec.iloc[0]['FOV(deg)'] * u.deg).value

        img_color = None
        encuesta_usada = "Ninguna"

        params = {
            "hips": host_info_aux.iloc[0]['SURVEY'],
            "ra": ra_real,
            "dec": dec_real,
            "fov": fov_en_grados,
            "width": 2000,
            "height": 2000,
            "projection": "TAN",
            "coordsys": "icrs",
            "format": "fits"
        }




        response = requests.get(url, params=params)
        #img_color = Image.open(io.BytesIO(response.content))
        #im2 = ax.imshow(img_color)
        #ax.axis("off")

        #fig.savefig(f"{host_info_aux.iloc[0]['NUMERO']}.{host_info_aux.iloc[0]['GALAXY']}.png")

        with fits.open(io.BytesIO(response.content)) as hdul:
            header = hdul[0].header
            wcs_imagen = WCS(header).celestial
            datos_crudos = hdul[0].data 
            datos_reordenados = np.moveaxis(datos_crudos, 0, -1)
            datos_imagen = datos_reordenados[:, :, :3]

        


        fig = plt.figure(figsize=(15, 15))

        ax = fig.add_subplot(1, 1, 1, projection=wcs_imagen)
        im = ax.imshow(datos_imagen, origin='lower') 
        ax.coords[0].set_axislabel('Ascensión Recta (RA)')
        ax.coords[1].set_axislabel('Declinación (DEC)')
        ax.coords.grid(color='white', alpha=0.5, linestyle='solid')



        f_px,f_py,RA_vG,DEC_vG = [],[],[],[]
        for i in range(len(selec)):
            Point = f"{selec['RA'].iloc()[i] + ' ' + selec['DEC'].iloc()[i]}" 
            coord = SkyCoord(Point, unit=(u.hourangle, u.deg), frame='icrs')
            RA_vG.append(coord.ra)
            DEC_vG.append(coord.dec)
            x, y = coord.to_pixel(wcs_imagen, origin=0)
            f_px.append(x)
            f_py.append(y)
        ax.plot(f_px, f_py, fillstyle='none',
                color='red', marker='o', markersize=10, linewidth = 0,
                label='GEHR')

        labels = np.array(selec['CLASS'])
        for i, label in enumerate(labels):
            ax.annotate(label, # The text label
                        (f_px[i], f_py[i]), # The point being annotated (xy)
                        textcoords="offset points", # How to position the text
                        xytext=(0, 10),color = 'orange', # Distance from the point (x, y)
                        ha='center') # Horizontal alignment
        

        # Busqueda de espectros de SDSS

        ancho_px = header['NAXIS1']
        alto_px = header['NAXIS2']
        escala_x = abs(header['CDELT1']) 
        escala_y = abs(header['CDELT2'])
        ancho_arcsec = ancho_px * escala_x * 3600
        alto_arcsec = alto_px * escala_y * 3600
        centro_x = (header['NAXIS1'] - 1) / 2.0
        centro_y = (header['NAXIS2'] - 1) / 2.0
        ra_cent, dec_cent = wcs_imagen.all_pix2world(centro_x, centro_y,0)
        center_im = SkyCoord(ra_cent,dec_cent, unit = (u.deg,u.deg),frame = 'icrs')

        data_releases = [17,18,19]
        dr_colors = ["#fd7af9c6","#7afc00c5","#ff7b00c5"]
        dr_tab_pos = ["upper left","lower left","lower right"]
        dr_markersize = [10,12,14]


        for DR_SDSS in range(len(data_releases)):
            xid_spec = SDSS.query_region(center_im, height = f'{alto_arcsec} arcsec', width = f'{ancho_arcsec} arcsec', spectro=True,data_release=data_releases[DR_SDSS])
            s_x,s_y,RA_vS,DEC_vS = [],[],[],[]
            if type(xid_spec) != type(None):
                    for i in range(len(xid_spec)):
                            coord_p = SkyCoord(xid_spec['ra'][i],xid_spec['dec'][i], unit=(u.deg, u.deg), frame='icrs')
                            RA_vS.append(coord_p.ra)
                            DEC_vS.append(coord_p.dec)
                            x, y = coord_p.to_pixel(wcs_imagen, origin=0)
                            s_x.append(x)
                            s_y.append(y)
            separation = 4
            catalog_matches = []
            if type(xid_spec) != type(None):
                    ax.plot(s_x, s_y,
                            markerfacecolor=dr_colors[DR_SDSS], marker='X', markersize=dr_markersize[DR_SDSS], linewidth = 0, markeredgewidth = 1,markeredgecolor='white',
                            label=f"SDSS spectra DR{data_releases[DR_SDSS]}",alpha=0.5)
                    sdss_view = SkyCoord(ra=RA_vS, dec=DEC_vS)
                    GEHR_view = SkyCoord(ra=RA_vG, dec=DEC_vG)
                    idx, d2d, d3d = GEHR_view.match_to_catalog_sky(sdss_view)
                    separation = 4
                    max_sep = separation * u.arcsec
                    sep_constraint = d2d < max_sep
                    c_matches = GEHR_view[sep_constraint]
                    catalog_matches = sdss_view[idx[sep_constraint]]
            if len(catalog_matches) >= 1:
                    data_table_onscreen = []
                    col_labels_tab = ['RA','DEC','PLATE', 'MJD','FIBER']
                    row_labels_tab = []
                    for b in range(len(idx)):
                        if d2d[b] < max_sep:
                            tmp_row = []
                            center_match = SkyCoord(sdss_view[idx[b]].ra,sdss_view[idx[b]].dec, unit = (u.deg,u.deg),frame = 'icrs')
                            xid_match = SDSS.query_region(center_match, height = f'4 arcsec', width = f'4 arcsec', spectro=True,data_release=data_releases[DR_SDSS])
                            row_labels_tab.append(selec['NOMBRE'].iloc[b])
                            tmp_row.append(xid_match['ra'][0])
                            tmp_row.append(xid_match['dec'][0])
                            tmp_row.append(xid_match['plate'][0])
                            tmp_row.append(xid_match['mjd'][0])
                            tmp_row.append(xid_match['fiberID'][0])
                            data_table_onscreen.append(tmp_row)
                    print(row_labels_tab)
                    print(data_table_onscreen)
                    ax.axis('on')
                    table = ax.table(cellText=data_table_onscreen,
                                    colLabels=col_labels_tab,
                                    rowLabels=row_labels_tab,
                                    colWidths=[0.17, 0.17,0.06,0.06,0.05],
                                    loc=f"{dr_tab_pos[DR_SDSS]}") # 'loc' specifies the table's position
                    table.auto_set_font_size(False)
                    table.set_fontsize(10)
                    table.scale(0.8, 1.0) # Scale the table size


        ax.legend(title = f"{host_info_aux.iloc[0]['GALAXY']} Max sep {separation} arcsec")
        print(f"¡Éxito! Imagen FITS obtenida. \n")
        with open(search_status, 'a') as f:
            f.write(f"{host_info_aux.iloc[0]['GALAXY']}\n")

        fig.savefig(f"{host_info_aux.iloc[0]['NUMERO']}.{host_info_aux.iloc[0]['GALAXY']}.png")


        